In [2]:
import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.io import loadmat
from scipy.signal import welch
from scipy.integrate import trapezoid
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


In [3]:
DATA_DIR = Path(r'F:\Github\pxsa\Digital-Biomarkers\Datasets\Multi-channel Wireless EEG Recordings of Young Adu')
OUTPUT_DIR = "./powerbi_exports"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FS = 128  # فرکانس نمونه‌برداری
EPOCH_SEC = 4  # طول هر پنجره به ثانیه
OVERLAP_SEC = 2  # همپوشانی پنجره‌ها
CHANNELS = ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4']

BANDS = {
    'Delta': (0.5, 4),
    'Theta': (4, 8),
    'Alpha': (8, 13),
    'Beta': (13, 30),
    'Gamma': (30, 45)
}

In [4]:
def compute_bandpowers(epoch_data, fs):
    """
    epoch_data: آرایه با ابعاد (samples, channels)
    خروجی: دیکشنری شامل توان نسبی هر باند برای هر کانال
    """
    feats = {}
    n_samples, n_ch = epoch_data.shape
    
    for ch_idx, ch_name in enumerate(CHANNELS):
        # محاسبه Power Spectral Density به روش Welch
        freqs, psd = welch(epoch_data[:, ch_idx], fs=fs, nperseg=min(n_samples, fs*2))
        total_power = trapezoid(psd, freqs) + 1e-10
        
        for band_name, (f_low, f_high) in BANDS.items():
            idx_band = np.logical_and(freqs >= f_low, freqs <= f_high)
            band_pow = trapezoid(psd[idx_band], freqs[idx_band])
            # توان نسبی (Relative Power)
            rel_power = band_pow / total_power
            feats[f"{ch_name}_{band_name}"] = rel_power
            
    return feats

In [5]:
mat_files = sorted(DATA_DIR.glob('*.mat'))

records = []
subject_metadata = []

for file_path in mat_files:
    fname = os.path.basename(file_path).replace('.mat', '')
    
    # تشخیص لیبل بر اساس نام فایل (ASub = Anxiety, ACSub = Control)
    if fname.startswith("ASub"):
        label = 1
        group = "Anxiety"
    elif fname.startswith("ACSub"):
        label = 0
        group = "Control"
    else:
        continue

    mat_content = loadmat(file_path)
    
    # پیدا کردن کلید اصلی ماتریس داده در فایل mat
    raw_data = None
    for k in mat_content:
        if not k.startswith("__") and isinstance(mat_content[k], np.ndarray):
            if mat_content[k].shape == (38400, 14) or mat_content[k].shape == (14, 38400):
                raw_data = mat_content[k]
                break
                
    if raw_data is None:
        continue
        
    if raw_data.shape[0] == 14:
        raw_data = raw_data.T  # تبدیل به (38400, 14)

    # قطعه‌بندی (Epoching)
    step_samples = int((EPOCH_SEC - OVERLAP_SEC) * FS)
    win_samples = int(EPOCH_SEC * FS)
    
    n_epochs = (len(raw_data) - win_samples) // step_samples + 1
    
    for ep in range(n_epochs):
        start = ep * step_samples
        end = start + win_samples
        window = raw_data[start:end, :]
        
        ep_feats = compute_bandpowers(window, FS)
        ep_feats['Subject_ID'] = fname
        ep_feats['Epoch_ID'] = ep
        ep_feats['Label'] = label
        ep_feats['Group'] = group
        records.append(ep_feats)
        
    subject_metadata.append({
        'Subject_ID': fname,
        'Group': group,
        'Label': label,
        'Total_Epochs': n_epochs
    })

df_all = pd.DataFrame(records)
df_meta = pd.DataFrame(subject_metadata)
print(f"تعداد کل نمونه‌های پنجره‌بندی شده: {len(df_all)}")

تعداد کل نمونه‌های پنجره‌بندی شده: 5662


In [6]:
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
# -------------------------------------------------------------
# ۴. مدل‌سازی یادگیری ماشین (Classification) - Subject-aware (no leakage)
# -------------------------------------------------------------
feat_cols = [c for c in df_all.columns if c not in ['Subject_ID', 'Epoch_ID', 'Label', 'Group']]
X = df_all[feat_cols].values
y = df_all['Label'].values
groups = df_all['Subject_ID'].values          # ← این خیلی مهمه

# ارزیابی ۵-فولد با در نظر گرفتن Subject (جلوگیری از data leakage)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)

y_pred = cross_val_predict(clf, X, y, cv=sgkf, groups=groups)
df_all['Prediction'] = y_pred

# آموزش نهایی روی کل داده فقط برای استخراج Feature Importance
clf.fit(X, y)
importances = clf.feature_importances_

df_importance = pd.DataFrame({
    'Feature': feat_cols,
    'Channel': [f.split('_')[0] for f in feat_cols],
    'Band': [f.split('_')[1] for f in feat_cols],
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

In [7]:
# -------------------------------------------------------------
# ۴. مدل‌سازی و مقایسه مدل‌ها (Subject-aware + Attention)
# -------------------------------------------------------------
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from xgboost import XGBClassifier
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

In [8]:

feat_cols = [c for c in df_all.columns if c not in ['Subject_ID', 'Epoch_ID', 'Label', 'Group', 'Prediction']]
X = df_all[feat_cols].values
y = df_all['Label'].values
groups = df_all['Subject_ID'].values

# -----------------------------
# 4.1 Classical models
# -----------------------------
models = {
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, 
                             use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=150, max_depth=5, random_state=42),
    "SVM_RBF": SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    "MLP": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300, random_state=42)
}

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

results = []
all_predictions = {}

print("Evaluating classical models (subject-aware CV)...")
for name, model in models.items():
    print(f"  → {name}")
    y_pred = cross_val_predict(model, X, y, cv=sgkf, groups=groups, n_jobs=-1)
    
    acc  = accuracy_score(y, y_pred)
    prec = precision_score(y, y_pred)
    rec  = recall_score(y, y_pred)
    f1   = f1_score(y, y_pred)
    
    results.append({
        "Model": name,
        "Accuracy": round(acc * 100, 2),
        "Precision": round(prec * 100, 2),
        "Recall": round(rec * 100, 2),
        "F1_Score": round(f1 * 100, 2)
    })
    all_predictions[name] = y_pred


Evaluating classical models (subject-aware CV)...
  → RandomForest
  → XGBoost
  → GradientBoosting
  → SVM_RBF
  → LogisticRegression
  → MLP


In [12]:
result = []

In [13]:

# -----------------------------
# 4.2 Attention-based model (PyTorch)
# -----------------------------
print("\nEvaluating Attention model...")

class EpochSequenceDataset(Dataset):
    def __init__(self, df, feat_cols, subject_ids, max_len=None):
        self.sequences = []
        self.labels = []
        self.subjects = []
        
        for sid in subject_ids:
            sub = df[df['Subject_ID'] == sid].sort_values('Epoch_ID')
            seq = sub[feat_cols].values.astype(np.float32)
            label = sub['Label'].iloc[0]
            self.sequences.append(seq)
            self.labels.append(label)
            self.subjects.append(sid)
        
        if max_len is None:
            self.max_len = max(len(s) for s in self.sequences)
        else:
            self.max_len = max_len
            
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        seq = self.sequences[idx]
        # pad / truncate
        if len(seq) > self.max_len:
            seq = seq[:self.max_len]
        else:
            pad = np.zeros((self.max_len - len(seq), seq.shape[1]), dtype=np.float32)
            seq = np.vstack([seq, pad])
        
        mask = (seq.sum(axis=1) != 0).astype(np.float32)  # 1 = real, 0 = pad
        return torch.tensor(seq), torch.tensor(mask), torch.tensor(self.labels[idx], dtype=torch.long)


class AttentionClassifier(nn.Module):
    def __init__(self, input_dim, d_model=64, nhead=4, num_layers=2, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=128,
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.attn_pool = nn.Linear(d_model, 1)          # simple attention pooling
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 2)
        )
        
    def forward(self, x, mask):
        # x: (B, T, F)
        x = self.input_proj(x)
        # create key_padding_mask (True = ignore)
        key_padding_mask = (mask == 0)
        x = self.transformer(x, src_key_padding_mask=key_padding_mask)
        
        # Attention pooling
        attn_weights = torch.softmax(self.attn_pool(x).squeeze(-1).masked_fill(key_padding_mask, -1e9), dim=1)
        x = torch.sum(x * attn_weights.unsqueeze(-1), dim=1)   # (B, d_model)
        
        return self.classifier(x)


def train_attention_model(train_df, val_df, feat_cols, epochs=50, batch_size=8, lr=1e-3, patience=8):
    train_subjects = train_df['Subject_ID'].unique()
    val_subjects   = val_df['Subject_ID'].unique()
    
    max_len = max(
        train_df.groupby('Subject_ID').size().max(),
        val_df.groupby('Subject_ID').size().max()
    )
    
    train_ds = EpochSequenceDataset(train_df, feat_cols, train_subjects, max_len)
    val_ds   = EpochSequenceDataset(val_df,   feat_cols, val_subjects,   max_len)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = AttentionClassifier(input_dim=len(feat_cols)).to(device)
    
    # ----- Class weights (important for imbalance) -----
    labels = train_df.groupby('Subject_ID')['Label'].first().values
    class_counts = np.bincount(labels)
    class_weights = 1.0 / class_counts
    class_weights = class_weights / class_weights.sum() * len(class_counts)
    weight_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
    
    criterion = nn.CrossEntropyLoss(weight=weight_tensor)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3
    )
    
    best_val_acc = 0.0
    best_state = None
    epochs_no_improve = 0
    
    for ep in range(epochs):
        # ---- Train ----
        model.train()
        for x, mask, yb in train_loader:
            x, mask, yb = x.to(device), mask.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(x, mask)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
            optimizer.step()
        
        # ---- Validation ----
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, mask, yb in val_loader:
                x, mask, yb = x.to(device), mask.to(device), yb.to(device)
                logits = model(x, mask)
                preds = logits.argmax(dim=1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)
        
        val_acc = correct / total if total > 0 else 0.0
        scheduler.step(val_acc)
        
        # Early stopping logic
        if val_acc > best_val_acc + 1e-4:          # small improvement threshold
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"    Early stopping at epoch {ep+1} (best val acc: {best_val_acc:.4f})")
                break
    
    # Load best weights
    if best_state is not None:
        model.load_state_dict(best_state)
    
    model.to(device)
    return model, max_len

# Subject-level 5-fold for Attention
unique_subjects = df_all['Subject_ID'].unique()
subject_labels  = df_all.groupby('Subject_ID')['Label'].first().reindex(unique_subjects).values

attn_preds = np.zeros(len(df_all), dtype=int)
attn_sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(attn_sgkf.split(unique_subjects, subject_labels, groups=unique_subjects)):
    train_subjects = unique_subjects[train_idx]
    val_subjects   = unique_subjects[val_idx]
    
    train_df = df_all[df_all['Subject_ID'].isin(train_subjects)]
    val_df   = df_all[df_all['Subject_ID'].isin(val_subjects)]
    
    # Scale features (important for NN)
    scaler = StandardScaler()
    train_df = train_df.copy()
    val_df   = val_df.copy()
    train_df[feat_cols] = scaler.fit_transform(train_df[feat_cols])
    val_df[feat_cols]   = scaler.transform(val_df[feat_cols])
    
    model, max_len = train_attention_model(train_df, val_df, feat_cols, epochs=35)
    
    # Predict on validation subjects
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()
    val_ds = EpochSequenceDataset(val_df, feat_cols, val_subjects, max_len)
    val_loader = DataLoader(val_ds, batch_size=8)
    
    with torch.no_grad():
        for i, (x, mask, _) in enumerate(val_loader):
            x, mask = x.to(device), mask.to(device)
            logits = model(x, mask)
            preds = logits.argmax(dim=1).cpu().numpy()
            
            # map back to original rows
            start = i * 8
            for j, pred in enumerate(preds):
                sid = val_subjects[start + j]
                mask_rows = df_all['Subject_ID'] == sid
                attn_preds[mask_rows] = pred

# Metrics for Attention
acc  = accuracy_score(y, attn_preds)
prec = precision_score(y, attn_preds)
rec  = recall_score(y, attn_preds)
f1   = f1_score(y, attn_preds)

results.append({
    "Model": "AttentionTransformer",
    "Accuracy": round(acc * 100, 2),
    "Precision": round(prec * 100, 2),
    "Recall": round(rec * 100, 2),
    "F1_Score": round(f1 * 100, 2)
})
all_predictions["AttentionTransformer"] = attn_preds

# -----------------------------
# 4.3 Save comparison & best model predictions
# -----------------------------
df_results = pd.DataFrame(results).sort_values("F1_Score", ascending=False)
df_results.to_csv(os.path.join(OUTPUT_DIR, "Model_Comparison.csv"), index=False)
print("\n=== Model Comparison ===")
print(df_results.to_string(index=False))

# Use the best classical model for the rest of the pipeline (or Attention if it wins)
best_model_name = df_results.iloc[0]["Model"]
print(f"\nBest model: {best_model_name}")

df_all['Prediction'] = all_predictions[best_model_name]

# Feature importance only for tree-based models
if best_model_name in ["RandomForest", "XGBoost", "GradientBoosting"]:
    best_model = models[best_model_name]
    best_model.fit(X, y)
    importances = best_model.feature_importances_
    df_importance = pd.DataFrame({
        'Feature': feat_cols,
        'Channel': [f.split('_')[0] for f in feat_cols],
        'Band': [f.split('_')[1] for f in feat_cols],
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)
else:
    # dummy importance for non-tree models
    df_importance = pd.DataFrame({
        'Feature': feat_cols,
        'Channel': [f.split('_')[0] for f in feat_cols],
        'Band': [f.split('_')[1] for f in feat_cols],
        'Importance': np.zeros(len(feat_cols))
    })


Evaluating Attention model...
    Early stopping at epoch 12 (best val acc: 0.7500)
    Early stopping at epoch 9 (best val acc: 0.4286)
    Early stopping at epoch 9 (best val acc: 0.6250)
    Early stopping at epoch 11 (best val acc: 0.8750)
    Early stopping at epoch 9 (best val acc: 0.7143)

=== Model Comparison ===
               Model  Accuracy  Precision  Recall  F1_Score
AttentionTransformer     71.05      80.00   69.57     74.42
AttentionTransformer     68.42      78.95   65.22     71.43
             SVM_RBF     61.94      67.80   70.70     69.22
        RandomForest     58.14      62.82   75.58     68.61
                 MLP     60.86      66.81   70.24     68.48
    GradientBoosting     59.22      64.72   71.72     68.04
             XGBoost     59.04      64.68   71.23     67.80
  LogisticRegression     55.99      63.78   63.15     63.46
AttentionTransformer     60.53      78.57   47.83     59.46

Best model: AttentionTransformer


In [13]:
# -------------------------------------------------------------
# ۵. ذخیره نتایج و فایل‌های Power BI (با پیش‌بینی Attention)
# -------------------------------------------------------------

# Use Attention predictions (best model)
df_all['Prediction'] = all_predictions["AttentionTransformer"]

# 1. Subject-level performance
subject_perf = df_all.groupby('Subject_ID').apply(
    lambda g: pd.Series({
        'Accuracy': accuracy_score(g['Label'], g['Prediction']),
        'Correct_Epochs': (g['Label'] == g['Prediction']).sum(),
        'Total_Epochs': len(g),
        'Predicted_Label': g['Prediction'].mode()[0],   # most frequent prediction
        'True_Label': g['Label'].iloc[0]
    })
).reset_index()

df_subject_summary = pd.merge(df_meta, subject_perf, on='Subject_ID')
df_subject_summary.to_csv(os.path.join(OUTPUT_DIR, "Subject_Summary.csv"), index=False)

# 2. Overall Metrics
overall_metrics = [
    {"Metric": "Accuracy",  "Value": round(accuracy_score(y, df_all['Prediction']) * 100, 2)},
    {"Metric": "Precision", "Value": round(precision_score(y, df_all['Prediction']) * 100, 2)},
    {"Metric": "Recall",    "Value": round(recall_score(y, df_all['Prediction']) * 100, 2)},
    {"Metric": "F1_Score",  "Value": round(f1_score(y, df_all['Prediction']) * 100, 2)},
]
pd.DataFrame(overall_metrics).to_csv(os.path.join(OUTPUT_DIR, "Overall_Metrics.csv"), index=False)

# 3. Confusion Matrix
cm = confusion_matrix(y, df_all['Prediction'])
df_cm = pd.DataFrame([
    {'Actual': 'Control', 'Predicted': 'Control', 'Count': int(cm[0, 0])},
    {'Actual': 'Control', 'Predicted': 'Anxiety', 'Count': int(cm[0, 1])},
    {'Actual': 'Anxiety', 'Predicted': 'Control', 'Count': int(cm[1, 0])},
    {'Actual': 'Anxiety', 'Predicted': 'Anxiety', 'Count': int(cm[1, 1])}
])
df_cm.to_csv(os.path.join(OUTPUT_DIR, "Confusion_Matrix.csv"), index=False)

# 4. Model Comparison (already created earlier)
df_results.to_csv(os.path.join(OUTPUT_DIR, "Model_Comparison.csv"), index=False)

# 5. Feature Importance
# Since Attention is not tree-based, we keep a placeholder.
# (You can later replace this with permutation importance if needed)
df_importance = pd.DataFrame({
    'Feature': feat_cols,
    'Channel': [f.split('_')[0] for f in feat_cols],
    'Band': [f.split('_')[1] for f in feat_cols],
    'Importance': 0.0
})
df_importance.to_csv(os.path.join(OUTPUT_DIR, "Feature_Importance.csv"), index=False)

# 6. Long-format bandpower (for group comparison charts)
bandpower_long = df_all.melt(
    id_vars=['Subject_ID', 'Group', 'Label', 'Prediction'],
    value_vars=feat_cols,
    var_name='Feature',
    value_name='Relative_Power'
)
bandpower_long['Channel'] = bandpower_long['Feature'].apply(lambda x: x.split('_')[0])
bandpower_long['Band'] = bandpower_long['Feature'].apply(lambda x: x.split('_')[1])
bandpower_long.drop(columns=['Feature']).to_csv(
    os.path.join(OUTPUT_DIR, "Channel_Bandpower_Long.csv"), index=False
)

# 7. Optional: Save the full epoch-level predictions
df_all[['Subject_ID', 'Epoch_ID', 'Group', 'Label', 'Prediction'] + feat_cols].to_csv(
    os.path.join(OUTPUT_DIR, "Epoch_Level_Predictions.csv"), index=False
)

print(f"\nAll files successfully saved in: {OUTPUT_DIR}")
print("Best model used for predictions: AttentionTransformer")
print(df_results.to_string(index=False))


All files successfully saved in: ./powerbi_exports
Best model used for predictions: AttentionTransformer
               Model  Accuracy  Precision  Recall  F1_Score
AttentionTransformer     76.32      75.00   91.30     82.35
AttentionTransformer     78.95      85.71   78.26     81.82
AttentionTransformer     68.42      72.00   78.26     75.00
             SVM_RBF     61.94      67.80   70.70     69.22
        RandomForest     58.14      62.82   75.58     68.61
                 MLP     60.86      66.81   70.24     68.48
    GradientBoosting     59.22      64.72   71.72     68.04
             XGBoost     59.04      64.68   71.23     67.80
  LogisticRegression     55.99      63.78   63.15     63.46


## Improved verstion of attention

In [12]:


feat_cols = [c for c in df_all.columns if c not in ['Subject_ID', 'Epoch_ID', 'Label', 'Group', 'Prediction']]
X = df_all[feat_cols].values
y = df_all['Label'].values
groups = df_all['Subject_ID'].values

# -----------------------------
# 4.1 Classical models
# -----------------------------
import math
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class AttentionClassifier(nn.Module):
    def __init__(self, input_dim, d_model=72, nhead=8, num_layers=2, dropout=0.3, max_len=300):
        super().__init__()
        
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout * 0.5)
        )
        
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos_encoder = PositionalEncoding(d_model, max_len=max_len + 1, dropout=dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 3,      # slightly smaller FFN
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True                     # Pre-Norm
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Attention Pooling
        self.attn_pool = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.Tanh(),
            nn.Linear(d_model // 2, 1, bias=False)
        )
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 2)
        )
        
        self._init_weights()
        
    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
                
    def forward(self, x, mask):
        """
        x:    (B, T, F)
        mask: (B, T)   → 1 = real, 0 = padding
        """
        B, T, _ = x.shape
        
        x = self.input_proj(x)
        
        # CLS token
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        
        # Extend mask for CLS
        cls_mask = torch.ones(B, 1, device=mask.device, dtype=mask.dtype)
        mask = torch.cat([cls_mask, mask], dim=1)
        
        x = self.pos_encoder(x)
        
        key_padding_mask = (mask == 0)
        x = self.transformer(x, src_key_padding_mask=key_padding_mask)
        
        # Attention Pooling
        attn_scores = self.attn_pool(x).squeeze(-1)
        attn_scores = attn_scores.masked_fill(key_padding_mask, -1e9)
        attn_weights = torch.softmax(attn_scores, dim=1)
        
        x = torch.sum(x * attn_weights.unsqueeze(-1), dim=1)
        
        return self.classifier(x)
    
def train_attention_model(train_df, val_df, feat_cols, epochs=50, batch_size=8, lr=1e-3, patience=8):
    train_subjects = train_df['Subject_ID'].unique()
    val_subjects   = val_df['Subject_ID'].unique()
    
    max_len = max(
        train_df.groupby('Subject_ID').size().max(),
        val_df.groupby('Subject_ID').size().max()
    )
    
    train_ds = EpochSequenceDataset(train_df, feat_cols, train_subjects, max_len)
    val_ds   = EpochSequenceDataset(val_df,   feat_cols, val_subjects,   max_len)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = AttentionClassifier(
        input_dim=len(feat_cols),
        d_model=72,
        nhead=8,
        num_layers=2,
        dropout=0.3
    ).to(device)
    
    # ----- Class weights (important for imbalance) -----
    labels = train_df.groupby('Subject_ID')['Label'].first().values
    class_counts = np.bincount(labels)
    class_weights = 1.0 / class_counts
    class_weights = class_weights / class_weights.sum() * len(class_counts)
    weight_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
    
    criterion = nn.CrossEntropyLoss(weight=weight_tensor)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3
    )
    
    best_val_acc = 0.0
    best_state = None
    epochs_no_improve = 0
    
    for ep in range(epochs):
        # ---- Train ----
        model.train()
        for x, mask, yb in train_loader:
            x, mask, yb = x.to(device), mask.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(x, mask)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
            optimizer.step()
        
        # ---- Validation ----
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, mask, yb in val_loader:
                x, mask, yb = x.to(device), mask.to(device), yb.to(device)
                logits = model(x, mask)
                preds = logits.argmax(dim=1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)
        
        val_acc = correct / total if total > 0 else 0.0
        scheduler.step(val_acc)
        
        # Early stopping logic
        if val_acc > best_val_acc + 1e-4:          # small improvement threshold
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"    Early stopping at epoch {ep+1} (best val acc: {best_val_acc:.4f})")
                break
    
    # Load best weights
    if best_state is not None:
        model.load_state_dict(best_state)
    
    model.to(device)
    return model, max_len

# Subject-level 5-fold for Attention
unique_subjects = df_all['Subject_ID'].unique()
subject_labels  = df_all.groupby('Subject_ID')['Label'].first().reindex(unique_subjects).values

attn_preds = np.zeros(len(df_all), dtype=int)
attn_sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(attn_sgkf.split(unique_subjects, subject_labels, groups=unique_subjects)):
    train_subjects = unique_subjects[train_idx]
    val_subjects   = unique_subjects[val_idx]
    
    train_df = df_all[df_all['Subject_ID'].isin(train_subjects)]
    val_df   = df_all[df_all['Subject_ID'].isin(val_subjects)]
    
    # Scale features (important for NN)
    scaler = StandardScaler()
    train_df = train_df.copy()
    val_df   = val_df.copy()
    train_df[feat_cols] = scaler.fit_transform(train_df[feat_cols])
    val_df[feat_cols]   = scaler.transform(val_df[feat_cols])
    
    model, max_len = train_attention_model(train_df, val_df, feat_cols, epochs=35)
    
    # Predict on validation subjects
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()
    val_ds = EpochSequenceDataset(val_df, feat_cols, val_subjects, max_len)
    val_loader = DataLoader(val_ds, batch_size=8)
    
    with torch.no_grad():
        for i, (x, mask, _) in enumerate(val_loader):
            x, mask = x.to(device), mask.to(device)
            logits = model(x, mask)
            preds = logits.argmax(dim=1).cpu().numpy()
            
            # map back to original rows
            start = i * 8
            for j, pred in enumerate(preds):
                sid = val_subjects[start + j]
                mask_rows = df_all['Subject_ID'] == sid
                attn_preds[mask_rows] = pred

# Metrics for Attention
acc  = accuracy_score(y, attn_preds)
prec = precision_score(y, attn_preds)
rec  = recall_score(y, attn_preds)
f1   = f1_score(y, attn_preds)

results.append({
    "Model": "AttentionTransformer",
    "Accuracy": round(acc * 100, 2),
    "Precision": round(prec * 100, 2),
    "Recall": round(rec * 100, 2),
    "F1_Score": round(f1 * 100, 2)
})
all_predictions["AttentionTransformer"] = attn_preds

# -----------------------------
# 4.3 Save comparison & best model predictions
# -----------------------------
df_results = pd.DataFrame(results).sort_values("F1_Score", ascending=False)
df_results.to_csv(os.path.join(OUTPUT_DIR, "Model_Comparison.csv"), index=False)
print("\n=== Model Comparison ===")
print(df_results.to_string(index=False))

# Use the best classical model for the rest of the pipeline (or Attention if it wins)
best_model_name = df_results.iloc[0]["Model"]
print(f"\nBest model: {best_model_name}")

df_all['Prediction'] = all_predictions[best_model_name]

# Feature importance only for tree-based models
if best_model_name in ["RandomForest", "XGBoost", "GradientBoosting"]:
    best_model = models[best_model_name]
    best_model.fit(X, y)
    importances = best_model.feature_importances_
    df_importance = pd.DataFrame({
        'Feature': feat_cols,
        'Channel': [f.split('_')[0] for f in feat_cols],
        'Band': [f.split('_')[1] for f in feat_cols],
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)
else:
    # dummy importance for non-tree models
    df_importance = pd.DataFrame({
        'Feature': feat_cols,
        'Channel': [f.split('_')[0] for f in feat_cols],
        'Band': [f.split('_')[1] for f in feat_cols],
        'Importance': np.zeros(len(feat_cols))
    })

    Early stopping at epoch 14 (best val acc: 0.8750)
    Early stopping at epoch 9 (best val acc: 0.5714)
    Early stopping at epoch 11 (best val acc: 0.6250)
    Early stopping at epoch 13 (best val acc: 0.5000)
    Early stopping at epoch 13 (best val acc: 0.8571)

=== Model Comparison ===
               Model  Accuracy  Precision  Recall  F1_Score
AttentionTransformer     76.32      75.00   91.30     82.35
AttentionTransformer     78.95      85.71   78.26     81.82
AttentionTransformer     68.42      72.00   78.26     75.00
             SVM_RBF     61.94      67.80   70.70     69.22
        RandomForest     58.14      62.82   75.58     68.61
                 MLP     60.86      66.81   70.24     68.48
    GradientBoosting     59.22      64.72   71.72     68.04
             XGBoost     59.04      64.68   71.23     67.80
  LogisticRegression     55.99      63.78   63.15     63.46

Best model: AttentionTransformer
